# Group Simulation for RL Agents and FNN Benchmark

This notebook orchestrates a systematic comparison of different modeling approaches. It trains multiple Reinforcement Learning (RL) agent configurations and a Feed-Forward Network (FNN) benchmark model, ensuring fair comparison by using identical random seeds. Finally, it evaluates all trained models on a unified test dataset to produce a comprehensive performance report with custom evaluation logic.

In [ ]:
# -------------------------------------------------------------------
# Cell 1: Setup Environment and Imports
# -------------------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
import time
import pandas as pd

# Add the project root to the Python path to resolve relative imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to Python path: {project_root}")

# Import all necessary functions and classes from the project package
from RL2gMCP.utils import Hyperparameters
from RL2gMCP.training import train_agent, calculate_primary_success_rate, calculate_secondary_power
from RL2gMCP.fnn_benchmark import find_optimal_graph_fnn
from RL2gMCP.evaluation import get_final_graph
from RL2gMCP.agent import Agent
from RL2gMCP.gMCP import gMCP

print("\nEnvironment setup complete. Ready to run experiments.")

In [ ]:
# -------------------------------------------------------------------
# Cell 2: Define Experiment Configurations
# -------------------------------------------------------------------

# Define partial graph constraints for the constrained experiments.
ALPHA_WEIGHT_FIX_EXAMPLE = [1.0, np.nan, np.nan, np.nan]
T_FIX_EXAMPLE = [
    [np.nan, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan],
    [np.nan, np.nan, np.nan, np.nan]
]

# Define all experimental variations, grouped by unconstrained and constrained
experiment_specs = [
    # --- Unconstrained Group ---
    {'name': 'RL-AvgPower-Full', 'type': 'rl', 'reward_type': 'avg_power', 'alpha_weight_fix': None, 't_fix': None},
    {'name': 'RL-PSR-Full', 'type': 'rl', 'reward_type': 'psr', 'alpha_weight_fix': None, 't_fix': None},
    {'name': 'FNN-Full', 'type': 'fnn', 'alpha_weight_fix': None, 't_fix': None},
    
    # --- Constrained Group ---
    {'name': 'RL-AvgPower-Constrained', 'type': 'rl', 'reward_type': 'avg_power', 'alpha_weight_fix': ALPHA_WEIGHT_FIX_EXAMPLE, 't_fix': T_FIX_EXAMPLE},
    {'name': 'RL-PSR-Constrained', 'type': 'rl', 'reward_type': 'psr', 'alpha_weight_fix': ALPHA_WEIGHT_FIX_EXAMPLE, 't_fix': T_FIX_EXAMPLE},
    {'name': 'FNN-Constrained', 'type': 'fnn', 'alpha_weight_fix': ALPHA_WEIGHT_FIX_EXAMPLE, 't_fix': T_FIX_EXAMPLE},
]

MASTER_SEED = 10001
print(f"Defined {len(experiment_specs)} experiments to run.")

In [ ]:
# -------------------------------------------------------------------
# Cell 3: Set Global Experiment Scenario Parameters
# -------------------------------------------------------------------

GLOBAL_SETTINGS = {
    'OUTPUT_DIR': "./results/",
    
    # --- General Training & Evaluation Parameters ---
    'LEARNING_RATE': 0.0001,
    'EPISODES': 3000, # For RL Agent training
    'BATCH_SIZE': 128,
    'NUM_SIMULATIONS': 10000, # For final evaluation
    
    # --- RL Agent Training & Architecture ---
    'AGENT_HIDDEN_LAYERS': [128, 64],
    'AGENT_DROPOUT_RATE': 0.2,
    'AGENT_TOP_K': 2, # Set to an integer (e.g., 2) to enforce sparsity

    # --- Experiment Scenario Settings ---
    'CORR_TYPE': "CS",
    'CORR_RHO': 0.0,
    'MARGINAL_POWER': [0.9, 0.8, 0.7, 0.6],
    'POWER_RANGE': 0.00,
    'REWARD_WEIGHTS': [0.6, 0.3, 0.05, 0.05],
    
    # --- Hierarchical Reward Settings ---
    'PRIMARY_ENDPOINTS': [1, 0, 0, 0],
    'PRIMARY_RULE': "PRIMARY",
    'PSR_THRESHOLD': 0.90,
    'PSR_PENALTY': 10.0,
    'W_PSR': 8.0,
    'W_SAP': 1.0
}

print("Global experiment scenario configured.")
os.makedirs(GLOBAL_SETTINGS['OUTPUT_DIR'], exist_ok=True)
print(f"Results will be saved to: {GLOBAL_SETTINGS['OUTPUT_DIR']}")

In [ ]:
# -------------------------------------------------------------------
# Cell 4: FNN Benchmark Process Settings
# -------------------------------------------------------------------

FNN_SETTINGS = {
    'FNN_SAMPLES': 10**3,          # Number of random graphs to generate for training FNN
    'FNN_LABEL_SIMS': 10**6,       # Number of p-value sims per graph to calculate its label (the paper uses 1,000,000)
    'FNN_EPOCHS': 10**4,            # Epochs for each candidate FNN architecture
    'COBYLA_MAX_ITER': 10**3        # Max iterations for the final fine-tuning step
}

print("FNN-specific process parameters configured.")

In [ ]:
# -------------------------------------------------------------------
# Cell 5: Helper Function for Dynamic Filename
# -------------------------------------------------------------------

def generate_filename(settings):
    m = len(settings['MARGINAL_POWER'])
    power_str = '_'.join(map(str, settings['MARGINAL_POWER']))
    cor_str = f"{settings['CORR_TYPE']}_{settings['CORR_RHO']}"
    primary_type_str = settings['PRIMARY_RULE']
    weight_str = '_'.join(map(str, settings['REWARD_WEIGHTS']))
    
    filename = f"m_{m}_power_{power_str}_cor_{cor_str}_{primary_type_str}_weight_{weight_str}.csv"
    return filename

print("Dynamic filename generator created.")
print(f"Example filename: {generate_filename(GLOBAL_SETTINGS)}")

In [ ]:
# -------------------------------------------------------------------
# Cell 6: Custom Evaluation Function (Post-Hoc Logic)
# -------------------------------------------------------------------

def run_custom_evaluation(all_graphs, eval_config, constrained_model_names):
    """A custom evaluation function to handle special Average Power calculation and timing."""
    device = eval_config.DEVICE
    m = len(eval_config.MARGINAL_POWER)
    num_simulations = eval_config.NUM_SIMULATIONS
    
    primary_mask = torch.tensor(eval_config.PRIMARY_ENDPOINTS, dtype=eval_config.DTYPE, device=device)
    original_weights = torch.tensor(eval_config.REWARD_WEIGHTS, dtype=eval_config.DTYPE, device=device)
    cov_matrix_eval = gMCP.create_covariance_matrix(m=m, corr_type=eval_config.CORR_TYPE, rho=eval_config.CORR_RHO)
    
    total_metrics = {g['name']: {'psr': 0.0, 'sap': 0.0, 'power': 0.0} for g in all_graphs}

    eval_batch_size = 2048
    with torch.no_grad():
        for i in range(0, num_simulations, eval_batch_size):
            current_batch_size = min(eval_batch_size, num_simulations - i)
            p_value_batch = gMCP.generate_p_values(
                batch_size=current_batch_size, 
                marginal_power=eval_config.MARGINAL_POWER, 
                power_range=eval_config.POWER_RANGE, 
                cov_matrix=cov_matrix_eval,
                sim_alpha=eval_config.ALPHA_0).to(dtype=eval_config.DTYPE, device=device)
            
            for graph_spec in all_graphs:
                name = graph_spec['name']
                decisions = torch.stack([gMCP.graphTest(graph_spec['alpha_weight'].to(device), graph_spec['T'].to(device), p, eval_config.ALPHA_0) for p in p_value_batch])
                
                if name in constrained_model_names:
                    power_weights = original_weights.clone()
                    if eval_config.ALPHA_WEIGHT_FIX is not None:
                        constrained_indices = ~np.isnan(eval_config.ALPHA_WEIGHT_FIX)
                        power_weights[constrained_indices] = 0.0
                else:
                    power_weights = original_weights
                
                W_scaled = power_weights / power_weights.sum() if power_weights.sum() > 0 else power_weights

                total_metrics[name]['psr'] += calculate_primary_success_rate(decisions, primary_mask, eval_config.PRIMARY_RULE, device) * current_batch_size
                total_metrics[name]['sap'] += calculate_secondary_power(decisions, primary_mask, original_weights, device) * current_batch_size
                total_metrics[name]['power'] += (decisions.float() @ W_scaled.to(device)).sum().item()

    avg_metrics = {name: {k: v / num_simulations for k, v in totals.items()} for name, totals in total_metrics.items()}
    
    report_data = []
    for graph_spec in all_graphs:
        name = graph_spec['name']
        metrics = avg_metrics[name]
        report_data.append({
            'Model': name,
            'PSR': metrics['psr'],
            'SAP': metrics['sap'],
            'Avg Power': metrics['power'],
            'Time (min)': graph_spec['training_time']
        })
    
    df = pd.DataFrame(report_data)
    
    model_order = [spec['name'] for spec in experiment_specs]
    df['Model'] = pd.Categorical(df['Model'], categories=model_order, ordered=True)
    df = df.sort_values('Model').reset_index(drop=True)
    
    numeric_cols = ['PSR', 'SAP', 'Avg Power', 'Time (min)']
    df[numeric_cols] = df[numeric_cols].round(3)
    
    print("\n--- Performance Comparison Report ---")
    print(df.to_string(index=False))
    
    filename = generate_filename(eval_config.__dict__)
    output_path = os.path.join(eval_config.OUTPUT_DIR, filename)
    df.to_csv(output_path, index=False)
    print(f"\nSuccessfully saved detailed report to: {output_path}")

print("Custom evaluation function created.")

In [ ]:
# -------------------------------------------------------------------
# Cell 7: Run Training Loop for All Models
# -------------------------------------------------------------------

print("--- Starting Model Training Loop ---")
training_start_time = time.time()
final_graphs_for_evaluation = []

for spec in experiment_specs:
    print("\n" + "="*50)
    print(f"BEGINNING EXPERIMENT: {spec['name']}")
    print("="*50)

    spec_start_time = time.time()
    
    torch.manual_seed(MASTER_SEED)
    np.random.seed(MASTER_SEED)

    config = Hyperparameters()
    for key, value in GLOBAL_SETTINGS.items():
        setattr(config, key, value)
    
    config.ALPHA_WEIGHT_FIX = spec.get('alpha_weight_fix', None)
    config.T_FIX = spec.get('t_fix', None)

    if spec['type'] == 'rl':
        config.REWARD_TYPE = spec['reward_type']
        trained_agent = train_agent(config, progress_gap=1000)
        alpha_fix = torch.tensor(config.ALPHA_WEIGHT_FIX, dtype=torch.float32) if config.ALPHA_WEIGHT_FIX is not None else None
        t_fix = torch.tensor(config.T_FIX, dtype=torch.float32) if config.T_FIX is not None else None
        alpha_weight, T = get_final_graph(trained_agent, alpha_fix, t_fix)
    
    elif spec['type'] == 'fnn':
        alpha_weight, T = find_optimal_graph_fnn(
            config=config,
            num_fnn_samples=FNN_SETTINGS['FNN_SAMPLES'],
            fnn_epochs=FNN_SETTINGS['FNN_EPOCHS'],
            cobyla_max_iter=FNN_SETTINGS['COBYLA_MAX_ITER'],
            fnn_label_sims=FNN_SETTINGS['FNN_LABEL_SIMS']
        )
    
    spec_end_time = time.time()
    training_time_min = (spec_end_time - spec_start_time) / 60

    final_graphs_for_evaluation.append({
        'name': spec['name'],
        'alpha_weight': alpha_weight.cpu(),
        'T': T.cpu(),
        'training_time': training_time_min
    })
    print(f"\nFINISHED EXPERIMENT: {spec['name']} in {training_time_min:.2f} minutes.")

training_end_time = time.time()
print("\n" + "="*50)
print(f"ALL MODELS TRAINED in {(training_end_time - training_start_time)/60:.2f} minutes.")
print("="*50)

In [ ]:
# -------------------------------------------------------------------
# Cell 8: Run Unified Final Evaluation
# -------------------------------------------------------------------

print("\n--- BEGINNING UNIFIED FINAL EVALUATION ---")

eval_config = Hyperparameters()
for key, value in GLOBAL_SETTINGS.items():
    setattr(eval_config, key, value)
eval_config.RANDOM_SEED = 99999

constrained_model_names = [spec['name'] for spec in experiment_specs if spec.get('alpha_weight_fix') is not None or spec.get('t_fix') is not None]

run_custom_evaluation(
    all_graphs=final_graphs_for_evaluation,
    eval_config=eval_config,
    constrained_model_names=constrained_model_names
)

print("\nGroup simulation finished.")

In [ ]:
# -------------------------------------------------------------------
# Cell 9: Display Final Learned Graphs (for inspection)
# -------------------------------------------------------------------

torch.set_printoptions(precision=4, sci_mode=False)

for graph_spec in final_graphs_for_evaluation:
    print("\n" + "-"*50)
    print(f"Final Graph for: {graph_spec['name']}")
    print("-"*50)
    
    print(f"Training Time: {graph_spec['training_time']:.2f} minutes")
    print("\nFinal Optimal Alpha Weights:")
    print(graph_spec['alpha_weight'].numpy())
    
    print("\nFinal Optimal T Matrix:")
    print(graph_spec['T'].numpy())